# 🌱 KrishiMitra - YOLOv11 Leaf Detection Training

This notebook guides you through training the single-class leaf detector (YOLOv11) on Google Colab with GPU support. Because training object detection models is computationally expensive, using a GPU instance in Google Colab will speed up the process dramatically compared to local CPU execution.

### **Steps to Follow:**
1. Open this notebook in Google Colab.
2. Connect to a GPU runtime (**Runtime -> Change runtime type -> T4 GPU** or higher).
3. Upload the generated dataset zip (`plantdoc_leaf.zip`) from your PC to Google Colab, or place it on Google Drive and mount it.

### Step 1: Install Dependencies
We need to install the `ultralytics` library to load, train, and export YOLOv11 models.

In [ ]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

### Step 2: Extract Dataset
Choose **Option A** if you upload the `plantdoc_leaf.zip` directly to the Colab session files, or **Option B** if you keep it on Google Drive.

#### **Option A: Direct Upload to Colab**
If you uploaded `plantdoc_leaf.zip` directly to the Colab storage panel on the left (under `/content/`), run the cell below to extract it.

In [ ]:
# Extract direct upload dataset
!unzip -q /content/plantdoc_leaf.zip -d /content/plantdoc_leaf
print("Dataset unzipped successfully to /content/plantdoc_leaf!")

#### **Option B: Using Google Drive**
If you mounted Google Drive and placed `plantdoc_leaf.zip` there, run these cells to mount Drive and extract the zip archive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adjust the path below to match where you saved plantdoc_leaf.zip on your Google Drive
ZIP_PATH = "/content/drive/MyDrive/KrishiMitra/plantdoc_leaf.zip"

!unzip -q {ZIP_PATH} -d /content/plantdoc_leaf
print("Dataset extracted from Google Drive successfully to /content/plantdoc_leaf!")

### Step 3: Create Colab YAML Configuration
YOLOv11 requires a configuration file describing the dataset class names and paths. We will create `dataset_colab.yaml` dynamically pointing to the Colab extraction directory.

In [ ]:
import yaml

dataset_config = {
    'path': '/content/plantdoc_leaf',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': {0: 'leaf'}
}

with open('/content/dataset_colab.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("Created dataset_colab.yaml successfully!")
with open('/content/dataset_colab.yaml', 'r') as f:
    print(f.read())

### Step 4: Train YOLOv11 Leaf Detector
We will load the pre-trained `yolo11n.pt` nano model as a starting baseline and fine-tune it on our leaf dataset for 100 epochs.

In [ ]:
from ultralytics import YOLO

# Load pre-trained nano model baseline
model = YOLO("yolo11n.pt")

# Train for 100 epochs on GPU
results = model.train(
    data="/content/dataset_colab.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,  # 0 indicates training on GPU
    workers=2,
    project="krishimitra_leaf_detection",
    name="yolov11_leaf"
)

### Step 5: Evaluate Training Results
After training, let's run validation on the best checkpoint and print the final metrics: Precision, Recall, mAP50, and mAP50-95.

In [ ]:
# Validate on the best checkpoint
metrics = model.val()

print("\n--- Model Performance Results ---")
print(f"Precision   : {metrics.results_dict['metrics/precision(B)']:.4f}")
print(f"Recall      : {metrics.results_dict['metrics/recall(B)']:.4f}")
print(f"mAP50       : {metrics.results_dict['metrics/mAP50(B)']:.4f}")
print(f"mAP50-95    : {metrics.results_dict['metrics/mAP50-95(B)']:.4f}")

### Step 6: Download the Trained Weights
Your trained weights will be saved at `krishimitra_leaf_detection/yolov11_leaf/weights/best.pt`.
Run the cells below to copy it to Google Drive or download it directly to your browser so you can copy it back to your local repository workspace.

In [ ]:
import shutil
from google.colab import files

trained_weights = "/content/krishimitra_leaf_detection/yolov11_leaf/weights/best.pt"

# Option A: Direct browser download
try:
    files.download(trained_weights)
    print("Triggered direct download for best.pt! Please rename it to yolov11_leaf.pt and copy it to saved_models/.")
except Exception as e:
    print(f"Direct download failed (ignore if using Google Drive mount): {e}")

# Option B: Save to Google Drive
try:
    drive_dest = "/content/drive/MyDrive/KrishiMitra/yolov11_leaf.pt"
    shutil.copy(trained_weights, drive_dest)
    print(f"Successfully copied weights to Google Drive path: {drive_dest}")
except Exception as e:
    print(f"Google Drive export failed: {e}")